# 06 — Website Content Localization with IP2Location.io

**Use case:** Suggest a visitor's language, currency, and regional storefront content based on the approximate location of their IP address. Let an explicit visitor selection override that suggestion.

**Learning goals**
1. Read sample website-visitor records and validate IP addresses.
2. Look up unique public IPv4 and IPv6 addresses with the IP2Location.io API.
3. Choose language, currency, region and localized copy using clear precedence rules.
4. Build a tiny storefront preview, summarize the results, and export a CSV.

> **Important:** The default `MODE = "demo"` uses **fabricated geolocation fixtures** and makes **zero API calls**. The mappings do not necessarily correspond to the actual geographic locations of the sample IPs. Set `MODE = "live"` to retrieve current IP2Location.io results. IP geolocation is approximate—not GPS—and should be used as a default, not an enforced identity or physical address.

## 1. Requirements

Python 3.9+ recommended. The notebook uses **requests**, **pandas**, and **matplotlib**. If a module is missing, run the following in a separate Jupyter cell:

```python
%pip install requests pandas matplotlib
```

API documentation: https://www.ip2location.io/ip2location-documentation

**Authentication:** Basic queries work without a key under the provider's keyless daily limit. For a registered plan, set the environment variable `IP2LOCATION_API_KEY`; the notebook sends it as a Bearer token and never prints it.

In [ ]:
import ipaddress
import os
import time
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import HTML, display

print('Dependencies loaded.')

## 2. Configuration and shared definitions

All constants used in both modes are defined here. Run the notebook **top to bottom** (Kernel → Restart & Run All). To use the real API, set `MODE = "live"` and optionally provide an API key via your environment.

- `MAX_LIVE_LOOKUPS` is a hard cap on unique public-IP calls for each run.
- `REQUEST_DELAY_SECONDS` introduces a short pause between calls.
- Country and currency settings represent an **example storefront configuration**. They are not a comprehensive country/locale database.

In [ ]:
MODE = "demo"  # "demo" (offline, synthetic) or "live" (real API)
API_URL = "https://api.ip2location.io/"
API_KEY = os.getenv("IP2LOCATION_API_KEY", "").strip()
INPUT_CSV = Path("sample_localization_visitors.csv")
OUTPUT_DIR = Path("outputs_notebook_06")
MAX_LIVE_LOOKUPS = 10
REQUEST_TIMEOUT_SECONDS = 15
REQUEST_DELAY_SECONDS = 0.3

assert MODE in {"demo", "live"}, "MODE must be 'demo' or 'live'."

# Regional policy of the example website. Edit to match your own supported markets.
MARKETS = {
    "US": {"name": "United States", "language": "en", "currency": "USD"},
    "GB": {"name": "United Kingdom", "language": "en", "currency": "GBP"},
    "SG": {"name": "Singapore", "language": "en", "currency": "SGD"},
    "MY": {"name": "Malaysia", "language": "ms", "currency": "MYR"},
    "DE": {"name": "Germany", "language": "de", "currency": "EUR"},
    "FR": {"name": "France", "language": "fr", "currency": "EUR"},
    "JP": {"name": "Japan", "language": "ja", "currency": "JPY"},
    "CN": {"name": "China", "language": "zh-cn", "currency": "CNY"},
    "BR": {"name": "Brazil", "language": "pt", "currency": "BRL"},
}
DEFAULT_COUNTRY = "US"

# *Illustrative storefront price points*, not live FX rates or converted prices.
DEMO_PRICE_BY_CURRENCY = {
    "USD": 19.00, "GBP": 15.00, "SGD": 25.00, "MYR": 89.00,
    "EUR": 17.00, "JPY": 2900, "CNY": 139.00, "BRL": 99.00,
}
CURRENCY_SYMBOL = {
    "USD": "$", "GBP": "£", "SGD": "S$", "MYR": "RM ",
    "EUR": "€", "JPY": "¥", "CNY": "¥", "BRL": "R$",
}

# Maintain localized copy in your CMS or translation catalogue in production.
COPY = {
    "en": {"title": "Welcome to our store", "cta": "Explore products", "shipping": "Check delivery options for {country}."},
    "ms": {"title": "Selamat datang ke kedai kami", "cta": "Terokai produk", "shipping": "Semak pilihan penghantaran untuk {country}."},
    "de": {"title": "Willkommen in unserem Shop", "cta": "Produkte entdecken", "shipping": "Lieferoptionen für {country} ansehen."},
    "fr": {"title": "Bienvenue dans notre boutique", "cta": "Découvrir les produits", "shipping": "Consultez les options de livraison pour {country}."},
    "ja": {"title": "当店へようこそ", "cta": "商品を見る", "shipping": "{country}への配送方法をご確認ください。"},
    "zh-cn": {"title": "欢迎光临我们的商店", "cta": "浏览商品", "shipping": "查看{country}的配送选项。"},
    "pt": {"title": "Bem-vindo à nossa loja", "cta": "Explorar produtos", "shipping": "Confira as opções de entrega para {country}."},
}
GEO_COLUMNS = [
    "country_code", "country_name", "region_name", "city_name",
    "api_language_code", "api_currency_code", "lookup_status", "lookup_error",
]

print(f"Mode: {MODE}; max real API calls: {MAX_LIVE_LOOKUPS}")
print("API key supplied:", bool(API_KEY))

## 3. Load sample website-visitor preferences

The CSV contains an artificial visitor ID, IP address, and optional choices for language, currency, and country/market. If you download only this notebook, it automatically creates the sample input file.

**Precedence:** Visitor choice → IP-based suggestion → website default. The `preferred_country` field models a user-selected storefront region, **not** a claim about where that person physically lives.

In [ ]:
SAMPLE_VISITORS = [
    ("V001", "8.8.8.8", "", "", "", "Default country-based experience"),
    ("V002", "1.1.1.1", "", "", "", "Different regional default"),
    ("V003", "9.9.9.9", "", "", "", "German-language page"),
    ("V004", "208.67.222.222", "", "", "", "French-language page"),
    ("V005", "2001:4860:4860::8888", "", "", "", "IPv6 example"),
    ("V006", "2606:4700:4700::1111", "", "", "", "Malay-language page"),
    ("V007", "114.114.114.114", "", "", "", "Chinese-language page"),
    ("V008", "8.26.56.26", "", "", "", "UK currency example"),
    ("V009", "8.8.8.8", "fr", "EUR", "", "Explicit language and currency override"),
    ("V010", "1.1.1.1", "ms", "MYR", "", "Personal preference overrides IP"),
    ("V011", "9.9.9.9", "", "USD", "", "Currency-only override"),
    ("V012", "192.168.1.10", "ja", "", "", "Private address: no API request"),
    ("V013", "not-an-ip", "", "", "", "Invalid address: no API request"),
    ("V014", "8.8.8.8", "", "", "JP", "Explicit market override"),
    ("V015", "8.26.56.26", "xx", "ZZZ", "", "Unsupported preference falls back"),
    ("V016", "1.1.1.1", "", "", "SG", "Selected market confirmed"),
]
INPUT_FIELDS = ["visitor_id", "ip", "preferred_language", "preferred_currency", "preferred_country", "scenario"]
if not INPUT_CSV.exists():
    pd.DataFrame(SAMPLE_VISITORS, columns=INPUT_FIELDS).to_csv(INPUT_CSV, index=False)
    print(f"Created {INPUT_CSV}")

visitors = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
required = set(INPUT_FIELDS) - {"scenario"}
missing = required - set(visitors.columns)
if missing:
    raise ValueError(f"Input CSV is missing columns: {sorted(missing)}")
for field in ("preferred_language", "preferred_currency", "preferred_country"):
    visitors[field] = visitors[field].fillna("").astype(str).str.strip()
visitors["ip"] = visitors["ip"].fillna("").astype(str).str.strip()
print(f"Loaded {len(visitors)} visitor rows from {INPUT_CSV}")
display(visitors.head(8))

## 4. Validate and deduplicate IP addresses

Do **not** send private, loopback, reserved, or malformed addresses to the public IP geolocation service. We validate addresses using Python's `ipaddress` module and make at most one lookup per distinct public IP in this run.

In [ ]:
def validate_ip(value):
    try:
        address = ipaddress.ip_address(str(value).strip())
    except ValueError:
        return "", "invalid_ip"
    if not address.is_global:
        return str(address), "non_public_ip"
    return str(address), "public_ip"

validated = visitors["ip"].map(validate_ip)
visitors["normalized_ip"] = validated.map(lambda item: item[0])
visitors["ip_status"] = validated.map(lambda item: item[1])
unique_public_ips = visitors.loc[visitors["ip_status"] == "public_ip", "normalized_ip"].drop_duplicates().tolist()
print(f"Rows: {len(visitors)}; unique public IPs: {len(unique_public_ips)}")
display(visitors[["visitor_id", "ip", "ip_status"]])

## 5. Offline synthetic fixture (demo mode only)

These country assignments are **fabricated for teaching**. The sample IPs are not guaranteed to be located in the listed countries. Most fixtures emulate the basic/free response; a few also emulate the optional nested `country.language` and `country.currency` metadata available on higher plans.

This entire cell is safe to skip in `MODE = "live"`: no live-mode function depends on variables created here.

In [ ]:
if MODE == "demo":
    SYNTHETIC_FIXTURES = {
        "8.8.8.8": {"country_code": "US", "country_name": "United States", "region_name": "California", "city_name": "Example City"},
        "1.1.1.1": {"country_code": "SG", "country_name": "Singapore", "region_name": "Central", "city_name": "Example City"},
        "9.9.9.9": {"country_code": "DE", "country_name": "Germany", "region_name": "Berlin", "city_name": "Example City"},
        "208.67.222.222": {"country_code": "FR", "country_name": "France", "region_name": "Île-de-France", "city_name": "Example City"},
        "2001:4860:4860::8888": {"country_code": "JP", "country_name": "Japan", "region_name": "Tokyo", "city_name": "Example City"},
        "2606:4700:4700::1111": {"country_code": "MY", "country_name": "Malaysia", "region_name": "Penang", "city_name": "Example City"},
        "114.114.114.114": {"country_code": "CN", "country_name": "China", "region_name": "Beijing", "city_name": "Example City"},
        "8.26.56.26": {"country_code": "GB", "country_name": "United Kingdom", "region_name": "England", "city_name": "Example City",
                       "country": {"language": {"code": "EN"}, "currency": {"code": "GBP"}}},
    }
    print(f"Loaded {len(SYNTHETIC_FIXTURES)} artificial geolocation fixtures; no requests will be sent.")
else:
    print("Live mode selected; demo fixtures are not loaded.")

## 6. API client and response normalization

Official endpoint: `GET https://api.ip2location.io/?ip=<address>&format=json`.

The free-plan response includes flat fields such as `country_code` and `city_name`. The richer nested `country.language.code` and `country.currency.code` can be absent, so the notebook does **not** require them. The API can return an `error` object even if an HTTP response succeeds; we check both. Errors and missing locations lead to sensible defaults rather than invented geolocations.

In [ ]:
def clean_code(value):
    return str(value or "").strip().upper() if value is not None else ""


def normalize_geolocation(payload, status="success", error=""):
    payload = payload if isinstance(payload, dict) else {}
    country_obj = payload.get("country") if isinstance(payload.get("country"), dict) else {}
    language_obj = country_obj.get("language") if isinstance(country_obj.get("language"), dict) else {}
    currency_obj = country_obj.get("currency") if isinstance(country_obj.get("currency"), dict) else {}
    return {
        "country_code": clean_code(payload.get("country_code")),
        "country_name": str(payload.get("country_name") or ""),
        "region_name": str(payload.get("region_name") or ""),
        "city_name": str(payload.get("city_name") or ""),
        "api_language_code": str(language_obj.get("code") or "").strip().lower(),
        "api_currency_code": clean_code(currency_obj.get("code")),
        "lookup_status": status,
        "lookup_error": str(error or "")[:180],
    }


def api_lookup_one(session, ip):
    params = {"ip": ip, "format": "json"}
    headers = {"Accept": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    try:
        response = session.get(API_URL, params=params, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        try:
            payload = response.json()
        except ValueError:
            return normalize_geolocation({}, status="response_not_json", error=f"HTTP {response.status_code}: non-JSON response"), False
        if not isinstance(payload, dict):
            return normalize_geolocation({}, status="invalid_response", error="Expected a JSON object"), False
        if isinstance(payload.get("error"), dict):
            api_error = payload["error"]
            code = str(api_error.get("error_code", "unknown"))
            msg = str(api_error.get("error_message", "API error"))
            stop = code == "10000" or response.status_code == 429
            return normalize_geolocation({}, status="api_error", error=f"{code}: {msg}"), stop
        if not response.ok:
            return normalize_geolocation({}, status="http_error", error=f"HTTP {response.status_code}"), response.status_code == 429
        if not payload.get("country_code"):
            return normalize_geolocation({}, status="country_unavailable", error="Response lacks country_code"), False
        return normalize_geolocation(payload), False
    except requests.RequestException as exc:
        return normalize_geolocation({}, status="network_error", error=f"{type(exc).__name__}: {exc}"), False

## 7. Run the geolocation lookup

In **demo mode**, each unique IP uses the fixture from Section 5 and consumes no API calls. In **live mode**, this cell makes at most `MAX_LIVE_LOOKUPS` real requests (one per unique public IP). Quota and authentication errors halt further lookups; remaining addresses receive an explicit skipped status.

In [ ]:
geo_by_ip = {}
if MODE == "demo":
    for ip in unique_public_ips:
        fixture = SYNTHETIC_FIXTURES.get(ip)
        geo_by_ip[ip] = normalize_geolocation(
            fixture or {},
            status="synthetic_demo" if fixture else "no_demo_fixture",
            error="" if fixture else "No synthetic fixture for this IP",
        )
else:
    stop_lookups = False
    with requests.Session() as session:
        for position, ip in enumerate(unique_public_ips):
            if stop_lookups:
                geo_by_ip[ip] = normalize_geolocation({}, status="skipped_quota_or_auth", error="Stopped after quota/auth error")
                continue
            if position >= MAX_LIVE_LOOKUPS:
                geo_by_ip[ip] = normalize_geolocation({}, status="skipped_limit", error="MAX_LIVE_LOOKUPS reached")
                continue
            result, stop_lookups = api_lookup_one(session, ip)
            geo_by_ip[ip] = result
            print(f"{position + 1:02d}: {ip} — {result['lookup_status']}")
            if not stop_lookups and position < len(unique_public_ips) - 1:
                time.sleep(REQUEST_DELAY_SECONDS)

print(f"Resolved/recorded {len(geo_by_ip)} unique public IPs in {MODE} mode.")
display(pd.DataFrame([{"ip": ip, **value} for ip, value in geo_by_ip.items()]))

## 8. Personalization rules: visitor choice wins

The **suggested market** comes from an explicit country selection, then a detected supported country, then a US default. The **language** and **currency** independently honor explicit selections, optional API metadata (when applicable), and configured market defaults. Unsupported choices are recorded in `preference_notes`; the notebook never silently treats them as authoritative.

Website copy and demo prices are maintained by the site, **not** translated or converted by IP2Location.io. Your application should offer visible language/market selectors, persist consented settings, and use an authoritative checkout service for prices, taxes, shipping, and exchange rates.

In [ ]:
def format_demo_price(currency):
    amount = DEMO_PRICE_BY_CURRENCY[currency]
    symbol = CURRENCY_SYMBOL[currency]
    digits = 0 if currency == "JPY" else 2
    return f"{symbol}{amount:,.{digits}f} ({currency})"


def personalize_visitor(visitor, geo):
    notes = []
    detected = geo["country_code"]
    selected_market = str(visitor.get("preferred_country") or "").strip().upper()
    preferred_language = str(visitor.get("preferred_language") or "").strip().lower().replace("_", "-")
    preferred_currency = str(visitor.get("preferred_currency") or "").strip().upper()

    if selected_market and selected_market in MARKETS:
        market = selected_market
        market_source = "visitor_choice"
    elif detected in MARKETS:
        market = detected
        market_source = "ip_suggestion"
    else:
        market = DEFAULT_COUNTRY
        market_source = "website_default"
    if selected_market and selected_market not in MARKETS:
        notes.append(f"Unsupported selected market: {selected_market}")

    market_config = MARKETS[market]
    api_language = geo.get("api_language_code", "").lower()
    api_currency = geo.get("api_currency_code", "").upper()
    use_geo_meta = market_source == "ip_suggestion"
    if preferred_language in COPY:
        language, language_source = preferred_language, "visitor_choice"
    elif use_geo_meta and api_language in COPY:
        language, language_source = api_language, "api_language_hint"
    else:
        language, language_source = market_config["language"], "market_default"
    if preferred_language and preferred_language not in COPY:
        notes.append(f"Unsupported language: {preferred_language}")

    if preferred_currency in DEMO_PRICE_BY_CURRENCY:
        currency, currency_source = preferred_currency, "visitor_choice"
    elif use_geo_meta and api_currency in DEMO_PRICE_BY_CURRENCY:
        currency, currency_source = api_currency, "api_currency_hint"
    else:
        currency, currency_source = market_config["currency"], "market_default"
    if preferred_currency and preferred_currency not in DEMO_PRICE_BY_CURRENCY:
        notes.append(f"Unsupported currency: {preferred_currency}")

    if geo["lookup_status"] not in {"success", "synthetic_demo"}:
        notes.append(f"Location unavailable ({geo['lookup_status']})")

    msg = COPY[language]
    return {
        "selected_market": market,
        "market_name": market_config["name"],
        "market_source": market_source,
        "display_language": language,
        "language_source": language_source,
        "display_currency": currency,
        "currency_source": currency_source,
        "welcome_title": msg["title"],
        "call_to_action": msg["cta"],
        "delivery_message": msg["shipping"].format(country=market_config["name"]),
        "illustrative_product_price": format_demo_price(currency),
        "preference_notes": "; ".join(notes),
    }

## 9. Combine visitor data, location and localized content

`lookup_status` always identifies how each record was geolocated. Invalid/non-public IPs are **not** sent to the API, and the website default applies unless the visitor has expressed a preference.

In [ ]:
completed_rows = []
for visitor in visitors.to_dict(orient="records"):
    if visitor["ip_status"] == "public_ip":
        geo = geo_by_ip[visitor["normalized_ip"]]
    else:
        geo = normalize_geolocation(
            {}, status=f"not_queried_{visitor['ip_status']}",
            error="IP is invalid or not publicly routable",
        )
    completed_rows.append({**visitor, **geo, **personalize_visitor(visitor, geo)})

results = pd.DataFrame(completed_rows)
summary_cols = [
    "visitor_id", "ip", "country_code", "lookup_status", "selected_market",
    "market_source", "display_language", "display_currency",
    "illustrative_product_price", "preference_notes",
]
display(results[summary_cols])

## 10. Small storefront preview (HTML)

This is a **static notebook preview**, not a deployed website. Its copy and product price are maintained by our example storefront. All dynamic text is HTML-escaped before display. Visitors should always be able to switch language/currency manually.

In [ ]:
def storefront_html(row):
    title = escape(str(row["welcome_title"]))
    cta = escape(str(row["call_to_action"]))
    delivery = escape(str(row["delivery_message"]))
    price = escape(str(row["illustrative_product_price"]))
    language = escape(str(row["display_language"]))
    country = escape(str(row["market_name"]))
    visitor = escape(str(row["visitor_id"]))
    return f"""
    <div style='border:1px solid #d9e1e7;border-radius:12px;padding:18px;margin:12px 0;font-family:Arial,sans-serif;max-width:620px'>
      <div style='font-size:12px;color:#667085'>Visitor {visitor} · Market: {country} · Language: {language}</div>
      <h3 style='margin:12px 0'>{title}</h3>
      <div style='font-size:21px;font-weight:bold'>{price}</div>
      <p style='color:#475467'>{delivery}</p>
      <div style='background:#183153;color:white;border-radius:7px;padding:9px 16px;display:inline-block'>{cta}</div>
      <p style='font-size:11px;color:#667085;margin-bottom:0'>Illustrative prices; review actual checkout terms before purchase.</p>
    </div>
    """

for row in results.head(4).to_dict(orient="records"):
    display(HTML(storefront_html(row)))

## 11. Summarize localization choices

Count final language, currency and market selections (including explicit overrides). These describe **site presentation choices**, not population distributions or the location of an individual.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results["display_language"].value_counts().sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Selected website language")
axes[0].set_xlabel("Visitor rows")
axes[0].set_ylabel("")
results["display_currency"].value_counts().sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Selected display currency")
axes[1].set_xlabel("Visitor rows")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

display(results.groupby(["selected_market", "display_language"], dropna=False).size().reset_index(name="visitors"))

## 12. Export results

Two files are created in `outputs_notebook_06`:

- `visitor_localization_results.csv`: per-visitor IP, geolocation lookup status, chosen language/currency, and localized content.
- `localization_summary.csv`: counts by selected market, language, and currency.

For real applications, avoid retaining raw IPs longer than necessary and follow applicable privacy rules.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
detail_csv = OUTPUT_DIR / "visitor_localization_results.csv"
summary_csv = OUTPUT_DIR / "localization_summary.csv"
results.to_csv(detail_csv, index=False, encoding="utf-8-sig")
summary = (
    results.groupby(["selected_market", "display_language", "display_currency"], dropna=False)
    .size().reset_index(name="visitor_rows")
    .sort_values("visitor_rows", ascending=False)
)
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print("Exported:", detail_csv.resolve())
print("Exported:", summary_csv.resolve())
display(summary)

## 13. Quick verification

These checks intentionally cover explicit preferences, market overrides, invalid/private IPs, and an unsupported language. They run in either mode and do not depend on synthetic fixture definitions.

In [ ]:
by_id = results.set_index("visitor_id")
assert len(results) == len(visitors)
assert set(GEO_COLUMNS).issubset(results.columns)
assert by_id.at["V009", "display_language"] == "fr"
assert by_id.at["V009", "display_currency"] == "EUR"
assert by_id.at["V014", "selected_market"] == "JP"
assert by_id.at["V014", "display_currency"] == "JPY"
assert by_id.at["V012", "lookup_status"] == "not_queried_non_public_ip"
assert by_id.at["V013", "lookup_status"] == "not_queried_invalid_ip"
assert "Unsupported language" in by_id.at["V015", "preference_notes"]
assert all(ip in geo_by_ip for ip in unique_public_ips)
print("All verification checks passed.")

## Practical deployment notes

- **Where to read the IP:** In a real web application, take the IP from a *trusted* web-server or load-balancer connection. Do not trust client-provided forwarding headers unless your reverse proxy is configured to sanitize them.
- **Preference priority:** A user-chosen language, currency or store region should take precedence over an IP geolocation suggestion. Remember their preference only with appropriate controls.
- **Location limitations:** Mobile networks, VPNs, proxies, carrier gateways and shared networks can cause incorrect location suggestions. IP geolocation is not GPS.
- **Response field availability:** Basic responses have country/region/city fields. The nested `country.language.code` and `country.currency.code` are optional subscription-dependent fields; this notebook handles both cases.
- **Translation vs localization:** The API reports metadata. It does not translate your storefront UI or calculate prices, tax, shipping eligibility, or current FX rates.
- **Rate limits:** Keyless requests have a documented daily quota. Use caching and an appropriate plan for production traffic; handle failures gracefully.

**Official references:**
- https://www.ip2location.io/ip2location-documentation
- https://www.ip2location.io/solutions/localization

**Next in the series:** Notebook 7 — Ad campaign geo-targeting and audience segmentation.